## 1. Загрузка данных, настройки и общие функции

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score

# ============================================================
# 1. НАСТРОЙКИ, ЗАГРУЗКА И ОБЩИЕ ФУНКЦИИ
# ============================================================

RANDOM_SEED = 42
ROOT = Path(".")
DATA_DIR = ROOT / "data"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Папка {DATA_DIR.resolve()} не найдена. "
    )

CAT_FEATURES = [
    "lead_source",
    "call_center",
    "region",
    "car_segment",
    "lead_channel",
    "user_tenure_bucket",
    "price_bucket",
    "assignment_hour",
    "assignment_weekday",
    "is_weekend",
]

DROP_COLS = [
    "lead_id",
    "user_id",
    "assignment_ts",
    "assignment_date",
    "target",
]


def load_base_data(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Загружает train/test и приводит даты и категории к правильному типу."""
    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")

    for frame in (train_df, test_df):
        frame["assignment_ts"] = pd.to_datetime(
            frame["assignment_ts"], errors="raise"
        )
        frame["assignment_date"] = pd.to_datetime(
            frame["assignment_date"], errors="raise"
        ).dt.normalize()

        for column in CAT_FEATURES:
            if column not in frame.columns:
                raise KeyError(f"В данных отсутствует категориальный признак: {column}")
            frame[column] = frame[column].fillna("__MISSING__").astype(str)

    # Проверяем ключи, чтобы merge и submission не портили строки.
    assert train_df["lead_id"].is_unique, "lead_id в train должен быть уникальным"
    assert test_df["lead_id"].is_unique, "lead_id в test должен быть уникальным"
    assert set(train_df["lead_id"]).isdisjoint(set(test_df["lead_id"])), (
        "lead_id из train и test пересекаются"
    )

    # Train сортируем по времени. Порядок test сохраняем исходным.
    train_df = train_df.sort_values(
        ["assignment_ts", "lead_id"]
    ).reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    return train_df, test_df


def daily_average_precision(
    validation_frame: pd.DataFrame,
    scores: np.ndarray,
) -> float:
    """
    Считает официальную локальную метрику:
    AP отдельно для каждой assignment_date, затем среднее по дням.
    """
    evaluation = validation_frame[["assignment_date", "target"]].copy()
    evaluation["score"] = np.asarray(scores, dtype=float)

    if len(evaluation) != len(scores):
        raise ValueError("Количество прогнозов не совпадает с числом строк")

    daily_scores: list[float] = []

    for _, day_frame in evaluation.groupby("assignment_date", sort=True):
        y_true = day_frame["target"].to_numpy()
        y_score = day_frame["score"].to_numpy()

        # В этих данных обычно есть оба класса. Обработка ниже нужна для надежности.
        positives = int(y_true.sum())
        if positives == 0:
            day_ap = 0.0
        elif positives == len(y_true):
            day_ap = 1.0
        else:
            day_ap = average_precision_score(y_true, y_score)

        daily_scores.append(float(day_ap))

    if not daily_scores:
        raise ValueError("Не удалось посчитать Daily AP: валидация пуста")

    return float(np.mean(daily_scores))


def make_date_holdout(
    frame: pd.DataFrame,
    validation_fraction: float = 0.20,
) -> tuple[pd.Series, pd.Series, np.ndarray]:
    """
    Делит train по ЦЕЛЫМ датам.
    Последние примерно 20% уникальных дней идут в validation.
    """
    unique_dates = np.array(
        sorted(frame["assignment_date"].dropna().unique())
    )

    if len(unique_dates) < 2:
        raise ValueError("Для time-based split нужно минимум две разные даты")

    validation_days_count = max(
        1,
        int(np.ceil(len(unique_dates) * validation_fraction)),
    )
    validation_days_count = min(validation_days_count, len(unique_dates) - 1)
    validation_dates = unique_dates[-validation_days_count:]

    train_mask = ~frame["assignment_date"].isin(validation_dates)
    validation_mask = frame["assignment_date"].isin(validation_dates)

    return train_mask, validation_mask, validation_dates


def validate_and_save_submission(
    test_frame: pd.DataFrame,
    scores: np.ndarray,
    output_path: str | Path,
) -> pd.DataFrame:
    """Проверяет формат submission и сохраняет его без индекса."""
    scores = np.asarray(scores, dtype=float)

    submission = pd.DataFrame(
        {
            "lead_id": test_frame["lead_id"].to_numpy(),
            "score": scores,
        }
    )

    assert list(submission.columns) == ["lead_id", "score"]
    assert len(submission) == len(test_frame)
    assert submission["lead_id"].is_unique
    assert submission["lead_id"].tolist() == test_frame["lead_id"].tolist()
    assert np.isfinite(submission["score"]).all()
    assert submission["score"].between(0.0, 1.0).all()

    submission.to_csv(output_path, index=False)
    print(f"Сохранен файл: {output_path}")
    return submission


print("Загрузка train.csv и test.csv...")
train, test = load_base_data(DATA_DIR)
print(f"train: {train.shape}, test: {test.shape}")

Загрузка train.csv и test.csv...
train: (13694, 119), test: (4306, 118)


## 2.baseline CatBoost

In [4]:
# ============================================================
# 2. BASELINE CATBOOST
# ============================================================

baseline_features = [
    column for column in train.columns if column not in DROP_COLS
]

baseline_train_mask, baseline_val_mask, baseline_val_dates = make_date_holdout(
    train,
    validation_fraction=0.20,
)

baseline_train_part = train.loc[baseline_train_mask].copy()
baseline_val_part = train.loc[baseline_val_mask].copy()

print("\nBASELINE")
print(
    "Validation dates:",
    [pd.Timestamp(date).date() for date in baseline_val_dates],
)
print(
    f"Train rows: {len(baseline_train_part)}, "
    f"validation rows: {len(baseline_val_part)}"
)

baseline_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="PRAUC",
    random_seed=RANDOM_SEED,
    task_type="CPU",
    allow_writing_files=False,
    verbose=100,
)

baseline_model.fit(
    baseline_train_part[baseline_features],
    baseline_train_part["target"],
    cat_features=CAT_FEATURES,
    eval_set=(
        baseline_val_part[baseline_features],
        baseline_val_part["target"],
    ),
    early_stopping_rounds=100,
    use_best_model=True,
)

baseline_val_predictions = baseline_model.predict_proba(
    baseline_val_part[baseline_features]
)[:, 1]

baseline_global_ap = average_precision_score(
    baseline_val_part["target"],
    baseline_val_predictions,
)
baseline_daily_ap = daily_average_precision(
    baseline_val_part,
    baseline_val_predictions,
)

print(f"Baseline global AP: {baseline_global_ap:.6f}")
print(f"Baseline Daily AP:  {baseline_daily_ap:.6f}")

# tree_count_ уже учитывает early stopping и use_best_model=True.
baseline_final_iterations = max(1, int(baseline_model.tree_count_))
print(f"Деревьев для финальной baseline-модели: {baseline_final_iterations}")

baseline_final_model = CatBoostClassifier(
    iterations=baseline_final_iterations,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    random_seed=RANDOM_SEED,
    task_type="CPU",
    allow_writing_files=False,
    verbose=False,
)

baseline_final_model.fit(
    train[baseline_features],
    train["target"],
    cat_features=CAT_FEATURES,
)

baseline_test_scores = baseline_final_model.predict_proba(
    test[baseline_features]
)[:, 1]

baseline_submission = validate_and_save_submission(
    test,
    baseline_test_scores,
    "catboost_baseline_submission.csv",
)


BASELINE
Validation dates: [datetime.date(2026, 4, 19), datetime.date(2026, 4, 20), datetime.date(2026, 4, 21), datetime.date(2026, 4, 22)]
Train rows: 10272, validation rows: 3422
0:	learn: 0.3553313	test: 0.3263930	best: 0.3263930 (0)	total: 200ms	remaining: 6m 39s
100:	learn: 0.5031259	test: 0.4358915	best: 0.4358915 (100)	total: 4.08s	remaining: 1m 16s
200:	learn: 0.5784839	test: 0.4613295	best: 0.4621098 (195)	total: 7.99s	remaining: 1m 11s
300:	learn: 0.6449576	test: 0.4790587	best: 0.4794839 (296)	total: 11.9s	remaining: 1m 7s
400:	learn: 0.7136252	test: 0.5073969	best: 0.5077475 (397)	total: 15.7s	remaining: 1m 2s
500:	learn: 0.7662938	test: 0.5212330	best: 0.5212330 (500)	total: 19.5s	remaining: 58.4s
600:	learn: 0.8074766	test: 0.5257348	best: 0.5261759 (598)	total: 23.2s	remaining: 54.1s
700:	learn: 0.8443448	test: 0.5296996	best: 0.5307073 (684)	total: 26.9s	remaining: 49.8s
800:	learn: 0.8749612	test: 0.5293247	best: 0.5308260 (726)	total: 30.5s	remaining: 45.6s
Stopped b

## 3. Признаки из events.csv без временной утечки

In [2]:
# ============================================================
# 3. РАСШИРЕННЫЕ ПРИЗНАКИ ИЗ EVENTS.CSV БЕЗ LEAKAGE
# ============================================================


def build_event_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    events_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Строит event-признаки только из событий event_ts < assignment_ts."""
    print("\nЗагрузка и обработка events.csv...")

    events = pd.read_csv(events_path)
    events["event_ts"] = pd.to_datetime(events["event_ts"], errors="coerce")

    train_work = train_df.copy()
    test_work = test_df.copy()

    train_work["_dataset"] = "train"
    test_work["_dataset"] = "test"
    train_work["_row_order"] = np.arange(len(train_work), dtype=np.int64)
    test_work["_row_order"] = np.arange(len(test_work), dtype=np.int64)

    all_leads = pd.concat(
        [train_work, test_work],
        ignore_index=True,
        sort=False,
    )

    lead_times = all_leads[["lead_id", "assignment_ts"]].copy()
    assert lead_times["lead_id"].is_unique

    events_valid = events.merge(
        lead_times,
        on="lead_id",
        how="inner",
        validate="many_to_one",
    )

    matched_count = len(events_valid)
    events_valid = events_valid.loc[
        events_valid["event_ts"].notna()
        & (events_valid["event_ts"] < events_valid["assignment_ts"])
    ].copy()

    print(f"Событий до назначения: {len(events_valid):,}")
    print(
        "Удалено событий после/в момент назначения: "
        f"{matched_count - len(events_valid):,}"
    )

    events_valid["hours_before_assign"] = (
        events_valid["assignment_ts"] - events_valid["event_ts"]
    ).dt.total_seconds() / 3600.0
    events_valid["event_day"] = events_valid["event_ts"].dt.normalize()
    events_valid = events_valid.sort_values(
        ["lead_id", "event_ts"]
    ).reset_index(drop=True)

    # Общая история событий.
    event_features = (
        events_valid.groupby("lead_id")
        .agg(
            event_total_count=("event_type", "size"),
            event_type_nunique=("event_type", "nunique"),
            event_day_nunique=("event_day", "nunique"),
            event_price_log_mean=("item_price_log", "mean"),
            event_price_log_std=("item_price_log", "std"),
            event_price_log_min=("item_price_log", "min"),
            event_price_log_max=("item_price_log", "max"),
            event_src_slot_mean=("src_slot", "mean"),
            event_src_slot_std=("src_slot", "std"),
            event_src_slot_nunique=("src_slot", "nunique"),
            event_ctx_seq_nunique=("ctx_seq", "nunique"),
            event_hours_since_last=("hours_before_assign", "min"),
            event_hours_since_first=("hours_before_assign", "max"),
        )
        .reset_index()
    )

    event_types = [
        "item_view",
        "search",
        "favorite",
        "chat_open",
        "call_click",
    ]

    # Контекст самого последнего события.
    last_event = (
        events_valid.groupby("lead_id", sort=False)
        .tail(1)[
            [
                "lead_id",
                "event_type",
                "ctx_seq",
                "src_slot",
                "item_price_log",
            ]
        ]
        .copy()
    )

    last_event = last_event.rename(
        columns={
            "src_slot": "last_event_src_slot",
            "item_price_log": "last_event_price_log",
        }
    )
    last_event["last_event_type"] = (
        last_event["event_type"].fillna("__NO_EVENT__").astype(str)
    )
    last_event["last_ctx_seq"] = (
        last_event["ctx_seq"].fillna("__NO_EVENT__").astype(str)
    )
    last_event["last_ctx_seq_code"] = pd.to_numeric(
        last_event["ctx_seq"].astype(str).str.extract(r"(\d+)")[0],
        errors="coerce",
    )

    for event_type in event_types:
        last_event[f"last_event_is_{event_type}"] = (
            last_event["event_type"].eq(event_type).astype("int8")
        )

    last_event = last_event.drop(columns=["event_type", "ctx_seq"])
    event_features = event_features.merge(
        last_event,
        on="lead_id",
        how="left",
        validate="one_to_one",
    )

    # Общие и per-type counts в нескольких временных окнах.
    windows = [
        (6, "6h"),
        (24, "24h"),
        (72, "72h"),
        (168, "7d"),
    ]

    for max_hours, label in windows:
        window_events = events_valid.loc[
            events_valid["hours_before_assign"] <= max_hours
        ]

        total_counts = (
            window_events.groupby("lead_id")
            .size()
            .rename(f"event_count_{label}")
            .reset_index()
        )

        type_counts = window_events.pivot_table(
            index="lead_id",
            columns="event_type",
            values="event_ts",
            aggfunc="count",
            fill_value=0,
        ).reindex(columns=event_types, fill_value=0)

        type_counts.columns = [
            f"event_count_{column}_{label}"
            for column in type_counts.columns
        ]

        event_features = event_features.merge(
            total_counts,
            on="lead_id",
            how="left",
            validate="one_to_one",
        )
        event_features = event_features.merge(
            type_counts.reset_index(),
            on="lead_id",
            how="left",
            validate="one_to_one",
        )

    # Counts по типам за всю доступную историю.
    all_type_counts = events_valid.pivot_table(
        index="lead_id",
        columns="event_type",
        values="event_ts",
        aggfunc="count",
        fill_value=0,
    ).reindex(columns=event_types, fill_value=0)
    all_type_counts.columns = [
        f"event_count_{column}_all"
        for column in all_type_counts.columns
    ]
    event_features = event_features.merge(
        all_type_counts.reset_index(),
        on="lead_id",
        how="left",
        validate="one_to_one",
    )

    # Время с последнего события каждого типа.
    type_recency = events_valid.pivot_table(
        index="lead_id",
        columns="event_type",
        values="hours_before_assign",
        aggfunc="min",
    ).reindex(columns=event_types)
    type_recency.columns = [
        f"event_hours_since_last_{column}"
        for column in type_recency.columns
    ]
    event_features = event_features.merge(
        type_recency.reset_index(),
        on="lead_id",
        how="left",
        validate="one_to_one",
    )

    # Экспоненциально затухающая активность: недавние события весят больше.
    for tau_hours in (24.0, 168.0):
        events_valid["_event_decay"] = np.exp(
            -events_valid["hours_before_assign"] / tau_hours
        )

        decay_features = events_valid.pivot_table(
            index="lead_id",
            columns="event_type",
            values="_event_decay",
            aggfunc="sum",
            fill_value=0,
        ).reindex(columns=event_types, fill_value=0)

        decay_features.columns = [
            f"event_decay_{column}_{int(tau_hours)}h"
            for column in decay_features.columns
        ]

        event_features = event_features.merge(
            decay_features.reset_index(),
            on="lead_id",
            how="left",
            validate="one_to_one",
        )

    all_leads = all_leads.merge(
        event_features,
        on="lead_id",
        how="left",
        validate="one_to_one",
    )

    all_leads["has_events"] = (
        all_leads["event_total_count"].notna().astype("int8")
    )

    for column in ["last_event_type", "last_ctx_seq"]:
        all_leads[column] = (
            all_leads[column].fillna("__NO_EVENT__").astype(str)
        )

    count_columns = [
        column
        for column in all_leads.columns
        if column.startswith("event_count_")
        or column.startswith("event_decay_")
        or column.startswith("last_event_is_")
    ]
    count_columns += [
        "event_total_count",
        "event_type_nunique",
        "event_day_nunique",
        "event_src_slot_nunique",
        "event_ctx_seq_nunique",
    ]
    count_columns = list(dict.fromkeys(count_columns))
    all_leads[count_columns] = all_leads[count_columns].fillna(0)

    all_leads["event_hours_since_last_missing"] = (
        all_leads["event_hours_since_last"].isna().astype("int8")
    )
    all_leads["event_log_hours_since_last"] = np.log1p(
        all_leads["event_hours_since_last"]
        .clip(lower=0, upper=24 * 90)
        .fillna(24 * 90)
    )
    all_leads["event_history_span_hours"] = (
        all_leads["event_hours_since_first"]
        - all_leads["event_hours_since_last"]
    ).clip(lower=0).fillna(0)

    train_enriched_df = (
        all_leads.loc[all_leads["_dataset"].eq("train")]
        .sort_values(["assignment_ts", "lead_id"])
        .drop(columns=["_dataset", "_row_order"])
        .reset_index(drop=True)
    )

    test_enriched_df = (
        all_leads.loc[all_leads["_dataset"].eq("test")]
        .sort_values("_row_order")
        .drop(
            columns=["_dataset", "_row_order", "target"],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    assert len(train_enriched_df) == len(train_df)
    assert len(test_enriched_df) == len(test_df)
    assert test_enriched_df["lead_id"].tolist() == test_df["lead_id"].tolist()

    added_columns = sorted(
        set(train_enriched_df.columns) - set(train_df.columns)
    )
    print(f"Добавлено event-признаков: {len(added_columns)}")

    return train_enriched_df, test_enriched_df


train_enriched, test_enriched = build_event_features(
    train,
    test,
    DATA_DIR / "events.csv",
)



Загрузка и обработка events.csv...
Событий до назначения: 238,581
Удалено событий после/в момент назначения: 16,124
Добавлено event-признаков: 71


## 4. CatBoost на обогащённых данных

In [6]:
# Промежуточную event-модель пропускаем.
# Финальная модель обучается в следующей большой клетке.



CATBOOST + EVENTS
Validation dates: [datetime.date(2026, 4, 19), datetime.date(2026, 4, 20), datetime.date(2026, 4, 21), datetime.date(2026, 4, 22)]
Количество признаков: 134
0:	learn: 0.4640210	test: 0.4705861	best: 0.4705861 (0)	total: 43.2ms	remaining: 1m 26s
100:	learn: 0.6009006	test: 0.5628886	best: 0.5631188 (99)	total: 3.97s	remaining: 1m 14s
200:	learn: 0.6574346	test: 0.5868537	best: 0.5869145 (198)	total: 7.98s	remaining: 1m 11s
300:	learn: 0.7080118	test: 0.6017907	best: 0.6017907 (300)	total: 11.8s	remaining: 1m 6s
400:	learn: 0.7601559	test: 0.6135764	best: 0.6136361 (391)	total: 15.5s	remaining: 1m 1s
500:	learn: 0.8076731	test: 0.6234084	best: 0.6234084 (500)	total: 19.3s	remaining: 57.8s
600:	learn: 0.8440397	test: 0.6298433	best: 0.6298433 (600)	total: 23.1s	remaining: 53.7s
700:	learn: 0.8744879	test: 0.6344512	best: 0.6349049 (696)	total: 26.8s	remaining: 49.6s
800:	learn: 0.9019709	test: 0.6371522	best: 0.6372954 (798)	total: 31s	remaining: 46.4s
900:	learn: 0.924

## 5. Advanced features и финальная модель


In [3]:
# ============================================================
# 5. ADVANCED FEATURES + ФИНАЛЬНЫЙ CATBOOST
# ============================================================

import gc


def add_advanced_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    new_features: dict[str, pd.Series] = {}

    def numeric(column: str) -> pd.Series:
        return (
            pd.to_numeric(result[column], errors="coerce")
            .fillna(0)
            .clip(lower=0)
        )

    def activity_trend(
        base_name: str,
        short_days: int,
        long_days: int,
    ) -> pd.Series:
        short_rate = numeric(f"{base_name}_{short_days}d") / short_days
        long_rate = numeric(f"{base_name}_{long_days}d") / long_days
        return np.log(
            (short_rate + 0.25) / (long_rate + 0.25)
        ).clip(-5, 5)

    def smoothed_rate(
        numerator: str,
        denominator: str,
        days: int,
    ) -> pd.Series:
        return np.log(
            (numeric(f"{numerator}_{days}d") + 0.5)
            / (numeric(f"{denominator}_{days}d") + 1.0)
        ).clip(-7, 3)

    # Циклические представления часа и дня недели.
    hour = pd.to_numeric(
        result["assignment_hour"], errors="coerce"
    ).fillna(-1)
    weekday = pd.to_numeric(
        result["assignment_weekday"], errors="coerce"
    ).fillna(-1)

    new_features["assignment_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    new_features["assignment_hour_cos"] = np.cos(2 * np.pi * hour / 24)
    new_features["assignment_weekday_sin"] = np.sin(
        2 * np.pi * weekday / 7
    )
    new_features["assignment_weekday_cos"] = np.cos(
        2 * np.pi * weekday / 7
    )

    # Изменение интенсивности активности относительно длинного окна.
    activity_bases = [
        "item_views",
        "item_favorites",
        "detail_expands",
        "photo_swipes",
        "seller_page_views",
        "search_views",
        "query_refinements",
        "similar_item_clicks",
        "saved_search_matches",
        "user_contacts",
        "chat_opens",
        "call_clicks",
        "leadgen_prev_assigned",
        "leadgen_prev_answered",
        "leadgen_prev_positive",
        "active_days_auto",
    ]

    for base_name in activity_bases:
        if f"{base_name}_90d" not in result.columns:
            continue

        for short_days, long_days in [
            (1, 7),
            (3, 14),
            (7, 30),
            (14, 90),
        ]:
            feature_name = (
                f"trend_{base_name}_{short_days}d_vs_{long_days}d"
            )
            new_features[feature_name] = activity_trend(
                base_name,
                short_days,
                long_days,
            )

    # Funnel-конверсии с устойчивым сглаживанием.
    rate_pairs = [
        ("item_favorites", "item_views"),
        ("detail_expands", "item_views"),
        ("photo_swipes", "item_views"),
        ("seller_page_views", "item_views"),
        ("query_refinements", "search_views"),
        ("similar_item_clicks", "search_views"),
        ("saved_search_matches", "search_views"),
        ("user_contacts", "item_views"),
        ("chat_opens", "user_contacts"),
        ("call_clicks", "user_contacts"),
        ("leadgen_prev_answered", "leadgen_prev_assigned"),
        ("leadgen_prev_positive", "leadgen_prev_assigned"),
        ("leadgen_prev_positive", "leadgen_prev_answered"),
    ]

    for numerator, denominator in rate_pairs:
        for days in (7, 30):
            numerator_column = f"{numerator}_{days}d"
            denominator_column = f"{denominator}_{days}d"

            if (
                numerator_column in result.columns
                and denominator_column in result.columns
            ):
                feature_name = (
                    f"rate_{numerator}_per_{denominator}_{days}d"
                )
                new_features[feature_name] = smoothed_rate(
                    numerator,
                    denominator,
                    days,
                )

    # Сопоставление текущей цены с историей событий.
    current_price = pd.to_numeric(
        result["item_price_log"], errors="coerce"
    )

    for statistic in ("mean", "min", "max"):
        history_price = pd.to_numeric(
            result[f"event_price_log_{statistic}"],
            errors="coerce",
        )
        new_features[f"price_diff_event_{statistic}"] = (
            current_price - history_price
        ).fillna(0)

    last_event_price = pd.to_numeric(
        result["last_event_price_log"], errors="coerce"
    )
    new_features["price_diff_last_event"] = (
        current_price - last_event_price
    ).fillna(0)
    new_features["price_history_missing"] = (
        result["event_price_log_mean"].isna().astype("int8")
    )

    # Доли типов событий в общей и недавней активности.
    total_events = np.maximum(numeric("event_total_count"), 1)

    for event_type in [
        "item_view",
        "search",
        "favorite",
        "chat_open",
        "call_click",
    ]:
        new_features[f"event_share_{event_type}_all"] = (
            numeric(f"event_count_{event_type}_all") / total_events
        )

        for label in ("24h", "72h", "7d"):
            total_window_events = np.maximum(
                numeric(f"event_count_{label}"),
                1,
            )
            new_features[f"event_share_{event_type}_{label}"] = (
                numeric(f"event_count_{event_type}_{label}")
                / total_window_events
            )

    history_days = np.maximum(
        pd.to_numeric(
            result["event_history_span_hours"],
            errors="coerce",
        ).fillna(0) / 24.0,
        1,
    )
    new_features["event_density_per_day"] = np.log1p(
        numeric("event_total_count") / history_days
    )

    new_features["log_views_per_day_alive"] = np.log1p(
        (numeric("item_views_90d") + 0.5)
        / (numeric("user_age_days") + 1.0)
    )
    new_features["log_contacts_per_active_day"] = np.log1p(
        (numeric("user_contacts_30d") + 0.5)
        / (numeric("user_active_days_30d") + 1.0)
    )

    source_feature_columns = [
        column
        for column in result.columns
        if column not in DROP_COLS
    ]
    new_features["feature_missing_count"] = (
        result[source_feature_columns]
        .isna()
        .sum(axis=1)
        .astype("int16")
    )

    result = pd.concat(
        [result, pd.DataFrame(new_features, index=result.index)],
        axis=1,
    ).copy()

    numeric_columns = result.select_dtypes(include=[np.number]).columns
    result[numeric_columns] = result[numeric_columns].replace(
        [np.inf, -np.inf],
        np.nan,
    )

    return result


print("\nДобавление advanced-признаков...")
train_final = add_advanced_features(train_enriched)
test_final = add_advanced_features(test_enriched)

final_features = [
    column
    for column in train_final.columns
    if column not in DROP_COLS
]

final_cat_features = CAT_FEATURES + [
    "last_event_type",
    "last_ctx_seq",
]

assert all(
    column in final_features
    for column in final_cat_features
)

print(f"Количество признаков: {len(final_features)}")
print(f"Категориальных признаков: {len(final_cat_features)}")

# Удаляем промежуточные CatBoost-модели, если предыдущие клетки запускались.
for model_name in [
    "baseline_model",
    "baseline_final_model",
    "event_model",
    "event_final_model",
    "optuna_final_model",
]:
    globals().pop(model_name, None)
gc.collect()

final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.04,
    depth=6,
    l2_leaf_reg=8.0,
    random_strength=0.8,
    bagging_temperature=0.7,
    loss_function="Logloss",
    random_seed=RANDOM_SEED,
    task_type="CPU",
    thread_count=4,
    border_count=32,
    rsm=0.8,
    allow_writing_files=False,
    verbose=100,
)

print("\nОбучение финальной модели...")
final_model.fit(
    train_final[final_features],
    train_final["target"],
    cat_features=final_cat_features,
)

raw_test_scores = final_model.predict_proba(
    test_final[final_features]
)[:, 1]

# Метрика считается внутри дня, поэтому переводим прогнозы
# в percentile-rank отдельно для каждой assignment_date.
rank_frame = pd.DataFrame(
    {
        "assignment_date": test_final["assignment_date"],
        "raw_score": raw_test_scores,
    }
)

day_rank_scores = (
    rank_frame.groupby("assignment_date")["raw_score"]
    .rank(method="average", pct=True)
    .to_numpy()
)

final_submission = validate_and_save_submission(
    test_final,
    day_rank_scores,
    "submission_enhanced.csv",
)

print(final_submission.head())
print("Готов файл submission_enhanced.csv")



Добавление advanced-признаков...
Количество признаков: 308
Категориальных признаков: 12

Обучение финальной модели...
0:	learn: 0.6700968	total: 189ms	remaining: 3m 9s
100:	learn: 0.3732867	total: 4.83s	remaining: 43s
200:	learn: 0.3436422	total: 9.45s	remaining: 37.6s
300:	learn: 0.3165949	total: 14s	remaining: 32.6s
400:	learn: 0.2902341	total: 18.6s	remaining: 27.7s
500:	learn: 0.2687157	total: 23.2s	remaining: 23.1s
600:	learn: 0.2517296	total: 27.9s	remaining: 18.5s
700:	learn: 0.2346092	total: 32.7s	remaining: 13.9s
800:	learn: 0.2206740	total: 37.7s	remaining: 9.36s
900:	learn: 0.2079687	total: 42.3s	remaining: 4.64s
999:	learn: 0.1954061	total: 46.8s	remaining: 0us
Сохранен файл: submission_enhanced.csv
                 lead_id     score
0  lead_97e409eb8f8c8246  0.270332
1  lead_55310edb4489f9e9  0.504640
2  lead_e7f653a2c6a7eee8  0.961054
3  lead_22f8e1cfc487ac20  0.235968
4  lead_48b638b839abfac3  0.532220
Готов файл submission_enhanced.csv
